# Task 2 - A Transformer that basecalls nanopore signals

Read the Task 2 background in the assignment PDF before starting. It covers the pore model,
the two strands, the adapter, and the shapes going in and out. This notebook is where you
build things, and it is meant to be worked in order.

## 0. Setup

In [ ]:
import time, numpy as np, torch
from torch import nn
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASES = "ACGT"
print("device:", DEVICE)

## 1. Look at the data

The generator below is the ground truth for the whole task: it turns a random strand into a
squiggle exactly as the PDF describes. `NOISE` is the main data slider and `hetero` is the other one, both used in Section 4.

In [ ]:
#@title The data generator - run it, nothing to change { display-mode: "form" }
K, DWELL, READ_LEN = 3, 8, 60
NOISE = 0.20                              # <-- Section 5 overrides this per run with noise=; leave it alone
NUM_KMERS = 4 ** K
TOTAL = READ_LEN * DWELL                  # signal length (fixed, even with dwell jitter)
GRID = np.arange(1, READ_LEN) * DWELL     # the even base boundaries (8, 16, ...)
COMP = np.array([3, 2, 1, 0])             # complement: A<->T, C<->G

rng = np.random.default_rng(42)
KMER_LEVEL = rng.normal(0, 1, NUM_KMERS).astype(np.float32)   # the pore model

def kmer_ids(seqs):
    B, n = seqs.shape
    p = np.pad(seqs, ((0, 0), (K - 1, 0)), mode="wrap")
    ids = np.zeros((B, n), dtype=np.int64)
    for j in range(K):
        ids = ids * 4 + p[:, j:j + n]
    return ids

def make_batch(batch, noise=None, gen=None, return_marker=False, hetero=0.0):
    '''signal (B, TOTAL), label (B, READ_LEN), orientation (B,), [marker base idx].

    A read is one strand of DNA pulled through the pore. Half the time it is the
    forward strand, half the time it is that strand's reverse complement. The
    current always comes from whichever strand is actually in the pore. The answer
    we want is always the forward strand.

    Reverse reads carry a leftover sequencing adapter, which the pore sees before any
    of the DNA and which shows up as a spike on the first base's samples.

    Two data sliders:
      noise   - Gaussian noise strength on the current
      hetero  - if >0, the noise level VARIES per base (in [1-h, 1+h] * noise)
    '''
    g = gen or rng; noise = NOISE if noise is None else noise
    templ = g.integers(0, 4, size=(batch, READ_LEN))           # the forward strand
    fwd = g.integers(0, 2, size=batch)                         # 1 = forward, 0 = reverse
    # whichever strand is physically in the pore is what sets the current
    in_pore = np.where(fwd[:, None] == 1, templ, COMP[templ][:, ::-1])
    lvl = KMER_LEVEL[kmer_ids(in_pore)]                        # (B, READ_LEN) level per base
    full = (np.arange(READ_LEN + 1) * DWELL)[None, :].repeat(batch, 0)
    t = np.arange(TOTAL)
    base_idx = np.clip((full[:, 1:, None] <= t[None, None, :]).sum(1), 0, READ_LEN - 1)
    sig = np.take_along_axis(lvl, base_idx, axis=1).astype(np.float32)     # (B, TOTAL)
    marker = np.where(fwd == 0, 0, -1)          # the adapter, at the read start
    sig += 6.0 * ((base_idx == marker[:, None]) & (fwd[:, None] == 0))     # adapter spike
    if hetero > 0:
        bscale = 1 + hetero * (2 * g.random((batch, READ_LEN)) - 1)
        psc = np.take_along_axis(bscale, base_idx, axis=1)
        sig += (g.normal(0, 1, sig.shape) * noise * psc).astype(np.float32)
    else:
        sig += g.normal(0, noise, size=sig.shape).astype(np.float32)
    label = templ                       # always report the forward strand
    out = [torch.from_numpy(sig), torch.from_numpy(label.copy()), torch.from_numpy(fwd)]
    if return_marker: out.append(torch.from_numpy(marker))
    return tuple(out)

### A forward read and a reverse read

Run this a few times. One feature separates the two, and *Q10* asks what it is. Look at how
large it is, how wide, whether it is always in the same place, and whether it ever shows up
on a forward read.

In [ ]:
#@title Plot a forward read and a reverse read { display-mode: "form" }
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
for ax, want in zip(axes, [1, 0]):
    while True:
        sig, lab, fwd = make_batch(1)
        if fwd[0] == want: break
    ax.plot(sig[0].numpy(), lw=.8)
    for i in range(READ_LEN): ax.axvline(i * DWELL, color="k", ls=":", alpha=.15)
    ax.set_title("FORWARD" if want else "REVERSE")
    ax.set_ylabel("current")
axes[-1].set_xlabel("signal sample"); plt.tight_layout(); plt.show()

### The same read, with both candidate answers

The pore reads whichever strand it captured, but the answer is always the forward strand.
The cell below draws one reverse capture with both strings underneath it, so you can see
which part of the signal fixes which base.

In [ ]:
#@title One reverse read, with both candidate answers { display-mode: "form" }
# The same bookkeeping, on one real reverse read. Each coloured pair is one answer base
# and the part of the signal that actually determines it.
while True:
    sig, lab, fwd = make_batch(1)
    if fwd[0] == 0: break

answer  = "".join(BASES[c] for c in lab[0].tolist())
in_pore = answer[::-1].translate(str.maketrans("ACGT", "TGCA"))     # the reverse complement

fig, ax = plt.subplots(2, 1, figsize=(11, 4.6), gridspec_kw={"height_ratios": [3, 1.5]})
ax[0].plot(sig[0].numpy(), lw=.8)
ax[0].set_ylabel("current"); ax[0].set_title("one REVERSE read")
for a in ax: a.set_xlim(-70, TOTAL)
ax[1].set_ylim(0, 1); ax[1].axis("off")

top, bot = [], []                       # keep the letter handles so we can recolour a few
for i in range(READ_LEN):
    x = (i + .5) * DWELL
    top.append(ax[1].text(x, .72, in_pore[i], ha="center", va="center",
                          fontsize=7, family="monospace"))
    bot.append(ax[1].text(x, .18, answer[i], ha="center", va="center",
                          fontsize=7, family="monospace"))
ax[1].text(-8, .72, "in the pore", ha="right", va="center", fontsize=8)
ax[1].text(-8, .18, "the answer",  ha="right", va="center", fontsize=8)

for i, colour, rad in [(3, "crimson", .18), (22, "seagreen", -.30), (48, "darkorange", .45)]:
    j = READ_LEN - 1 - i                                  # where that base's current sits
    for t in (top[j], bot[i]):
        t.set_color(colour); t.set_weight("bold"); t.set_fontsize(9)
    ax[1].annotate("", xy=((i + .5) * DWELL, .28), xytext=((j + .5) * DWELL, .64),
                   arrowprops=dict(arrowstyle="->", lw=1.3, color=colour, alpha=.9,
                                   connectionstyle=f"arc3,rad={rad}"))
plt.tight_layout(); plt.show()

## 2. First attempt: the CNN you already know (TODO-1)

Task 1's CNN ended by flattening everything down to a single number for the whole sequence.
Here you need an answer at *every* position, so nothing is ever flattened: 480 samples go
in, become 60 positions, and stay 60 positions until the final layer turns each one into its
4 scores.

| | |
|---|---|
| `stem` (given) | each base's 8 samples into one token of width `C`, giving `(batch, C, 60)` |
| `body` (yours) | mix each token with its neighbours: `Conv1d(C, C, kernel, padding=kernel//2)` then `ReLU`, repeated. Shape unchanged. |
| `head` (yours) | `Conv1d(C, 4, kernel_size=1)`, a `Linear` applied at each position on its own, giving `(batch, 4, 60)` |

`forward` is written for you, transposes included.

In [ ]:
class CNNBasecaller(nn.Module):
    """The Task 1 architecture, adapted: one label per base instead of one per sequence."""
    def __init__(self, C=112, layers=3, kernel=5):
        super().__init__()
        # given: the same tokenizer trick - one token per base
        self.stem = nn.Conv1d(1, C, kernel_size=DWELL, stride=DWELL)

        # TODO-1a: the body
        body = []
        for _ in range(layers):
            body += [nn.Conv1d(C, C, kernel, padding=kernel // 2), nn.ReLU()]
        self.body = nn.Sequential(*body)

        # TODO-1b: the head
        self.head = nn.Conv1d(C, 4, kernel_size=1)

    def forward(self, sig):                    # sig (batch, READ_LEN*DWELL)
        x = self.stem(sig[:, None, :])         # -> (batch, C, READ_LEN)
        x = self.body(x)                       # -> (batch, C, READ_LEN)
        x = self.head(x)                       # -> (batch, 4, READ_LEN)
        return x.transpose(1, 2)               # -> (batch, READ_LEN, 4)



### CNN Training

Task 1 paired `sigmoid` with `BCEWithLogitsLoss` for a yes/no answer. The four-way version
is `softmax` with `nn.CrossEntropyLoss`, and `train` below already uses it. Like
`BCEWithLogitsLoss` it applies the softmax internally, so your head stays plain with no
softmax inside it. The called base is the largest of the four scores (`.argmax(-1)`).

`evaluate` reports accuracy **split by orientation**. Watch the two columns, not just the
overall number - ***Q11*** asks you to report both.

`train` reports at every checkpoint as it goes, and returns a dict: `acc`, `fwd`, `rev`,
`params`, and `hist`. `hist` is the loss and accuracy recorded at each checkpoint, which is
what you plot in Section 4 to compare one architecture with another.

In [ ]:
#@title evaluate() and train() - run it, nothing to change { display-mode: "form" }
def evaluate(model, n=4000, seed=1234, **data):
    # fixed generator: the same model always scores the same, on the same reads
    model.eval()
    with torch.no_grad():
        sig, lab, fwd = make_batch(n, gen=np.random.default_rng(seed), **data)
        pred = model(sig.to(DEVICE)).argmax(-1).cpu()
    ok = (pred == lab).float().mean(1)
    return (pred == lab).float().mean().item(), ok[fwd == 1].mean().item(), ok[fwd == 0].mean().item()

def train(model, name="model", steps=3000, batch=128, lr=1.5e-3, seed=0, every=500, **data):
    # **data forwards the sliders (noise=, hetero=) to make_batch
    # NOTE: this seeds the shuffling, NOT the weights - the model was already built by
    # the time we get here. To fix the weights too, seed before you build it:
    #   torch.manual_seed(0); model = Basecaller(...)
    torch.manual_seed(seed); model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.CrossEntropyLoss()
    gen = np.random.default_rng(7); t0 = time.time()
    hist, run = [], torch.zeros((), device=DEVICE)
    for s in range(1, steps + 1):
        sig, lab, _ = make_batch(batch, gen=gen, **data)
        loss = lossf(model(sig.to(DEVICE)).reshape(-1, 4), lab.reshape(-1).to(DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.detach()                       # kept on the GPU, no sync every step
        if s % every == 0:                         # checkpoint: record it and say so
            a_, f_, r_ = evaluate(model, **data)
            hist.append(dict(step=s, loss=(run / every).item(), acc=a_, fwd=f_, rev=r_))
            print(f"   step {s:>5}/{steps}   loss {hist[-1]['loss']:.3f}   "
                  f"acc {a_:.3f} (fwd {f_:.3f} | rev {r_:.3f})", flush=True)
            run = torch.zeros((), device=DEVICE); model.train()
    acc, af, ar = evaluate(model, **data)          # evaluate in the SAME regime it trained on
    print(f"{name:16s} params={sum(p.numel() for p in model.parameters()):>8,}  "
          f"acc={acc:.3f}  (forward {af:.3f} | reverse {ar:.3f})  {time.time()-t0:.0f}s")
    return dict(name=name, acc=acc, fwd=af, rev=ar, hist=hist,
                params=sum(p.numel() for p in model.parameters()))

In [ ]:
cnn = CNNBasecaller()
train(cnn, "CNN baseline")

## 3. The fix: attention (TODO-2 and TODO-3)

Whatever you concluded in Section 2, the fix is not a bigger version of the same thing. What
you need is an architecture where a position can draw on any other position, however far
away, and decide for itself which one. That is what self-attention does.

The five blocks below are given. You create them with a size and use them, and nothing
inside them is yours to change. `MultiHeadSelfAttention` is written out rather than imported
so you can see the mechanism from the tutorial.

In [ ]:
#@title The five building blocks - run it, nothing to change { display-mode: "form" }
class Tokenizer(nn.Module):
    "Collapse each base's DWELL signal samples into one token vector (a strided conv)."
    def __init__(self, d_model):
        super().__init__()
        self.conv = nn.Conv1d(1, d_model, kernel_size=DWELL, stride=DWELL)
    def forward(self, sig):                 # sig (B, READ_LEN*DWELL)
        return self.conv(sig[:, None, :]).transpose(1, 2)     # (B, READ_LEN, d_model)

class PositionalEncoding(nn.Module):
    "Add a fixed sinusoidal position signature so attention can tell positions apart."
    def __init__(self, d_model, max_len=READ_LEN):
        super().__init__()
        assert d_model % 2 == 0, f"d_model ({d_model}) must be even"
        pos = torch.arange(max_len)[:, None]
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe)
    def forward(self, x):
        return x + self.pe[None, :x.size(1)]

class MultiHeadSelfAttention(nn.Module):
    "Every token looks at every token: softmax(Q Kᵀ / sqrt(dk)) V, from scratch."
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, f"d_model ({d_model}) must divide evenly by n_heads ({n_heads})"
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x, return_attn=False):
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.h, self.dk).transpose(1, 2) for t in (q, k, v)]
        att = (q @ k.transpose(-2, -1) / self.dk ** 0.5).softmax(dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, T, d)
        return (self.out(y), att) if return_attn else self.out(y)

class FeedForward(nn.Module):
    "Per-token 2-layer MLP that mixes features (not positions)."
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class EncoderBlock(nn.Module):
    "One encoder: attention, then feed-forward - each wrapped in a residual + LayerNorm."
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # attention + residual connection
        x = x + self.ff(self.ln2(x))       # feed-forward + residual connection
        return x

**`TODO-2`.** Wire the blocks into the full model: a `Tokenizer`, a
`PositionalEncoding`, `n_layers` `EncoderBlock`s, and a `Linear` head. The constructor
arguments are your architecture decisions: `d_model` (token width), `n_heads`, `n_layers`,
`d_ff` (feed-forward width), and `use_pe` (positional encoding on/off, for Section 5).

In [ ]:
class Basecaller(nn.Module):
    def __init__(self, d_model=96, n_heads=4, n_layers=3, d_ff=128, use_pe=True):
        super().__init__()
        # TODO-2: build the stack from the bricks (these args are your design choices)
        #   self.tokenizer : Tokenizer(...)
        #   self.pos       : PositionalEncoding(...)  (only if use_pe, otherwise None)
        #   self.blocks    : nn.ModuleList of n_layers EncoderBlock(...)
        #   self.head      : nn.Linear(...) mapping each token to 4 base logits
        #
        # Sanity check: with the default sizes above, `train` should print roughly
        # 190,000 parameters. A much smaller number means some of your layers were not
        # registered - which is what `nn.ModuleList` is for, and a plain list is not.
        self.tokenizer = ...
        self.pos = ...
        self.blocks = ...
        self.head = ...
    def forward(self, sig):
        x = self.tokenizer(sig)
        if self.pos is not None: x = self.pos(x)
        for block in self.blocks: x = block(x)
        return self.head(x)                       # (B, READ_LEN, 4)

**`TODO-3`.** Instantiate your `Basecaller` with sizes you choose and
train it on the *same* data as the CNN. Compare the two accuracy columns with what the CNN
managed - that comparison is ***Q12***. Note the parameter counts too: the two models are
deliberately about the same size, so what changes is the architecture, not the capacity.

In [ ]:
# TODO-3: instantiate YOUR Basecaller (pick d_model / n_heads / n_layers / d_ff) and train it
model = Basecaller(d_model=..., n_heads=..., n_layers=..., d_ff=...)
train(model, "Transformer")

## 4. Harder data, bigger models (TODO-4)

Vary the model sizes and the two data settings, and keep the runs you need to justify a
choice of architecture. `train` hands back `hist`, so you can plot learning curves as well as
final numbers. Then train that choice in the graded regime, `noise=0.5, hetero=0.6`.
***Q13*** asks for the architecture, the plots behind it, and what the sweep told you.

You do not need long runs to compare sizes. The gap between a small and a large model is
clear after about `steps=1000`, and training longer mostly lifts both together. Sweep short,
then spend one long run on the graded regime.

In [ ]:
# TODO-4: compare at least two model sizes across noise levels, and keep the results.
#         `train` returns a dict, so r["acc"] is the final number and r["hist"] is the curve.
for noise in [0.3, 0.5, 0.7]:
    r = train(Basecaller(d_model=..., n_heads=..., n_layers=..., d_ff=...),
              f"noise={noise}", steps=1000, noise=noise)

# then the regime you are ranked on:
train(Basecaller(d_model=..., n_heads=..., n_layers=..., d_ff=...),
      "graded regime", noise=0.5, hetero=0.6)

## 5. Ablation: turn positional encoding off (TODO-5)

Write your prediction down before you run this. Does removing the positional encoding cost
forward and reverse captures the same amount? Think about what each one actually needs.

Then rebuild with `use_pe=False`, keeping every other size fixed, and retrain. ***Q14*** asks
for the prediction, the numbers, and the difference between them.

In [ ]:
# TODO-5: rebuild with use_pe=False and compare. Keep every other size identical to the
#         model you trained in Section 3 - change one thing at a time, or you are
#         comparing two differences at once.
m = Basecaller(d_model=..., n_heads=..., n_layers=..., d_ff=..., use_pe=False)
train(m, "no-PE")

## 6. What did it learn?

Section 5 broke something and watched the damage. This section looks at the model without
touching it.

For one read we can plot, for every position the model computes, how much weight it put on
every position it could have looked at. That is one square grid per head. Row $i$ is the
position being computed, column $j$ is the position it looked at, and brightness is the
weight. Every row sums to 1, so an even split over 60 positions would be 0.017 everywhere,
and a bright cell means that one position got a large share of the attention.

Three shapes are worth recognising, and the cell below draws them so you know what you are
looking at before you see the real thing:

- a bright **main diagonal**: works locally, each position looking at itself and its
  neighbours, which is what resolving a 3-mer needs
- a bright **vertical stripe**: every position looking at the same single place
- a bright **anti-diagonal**: each position looking at its mirror, the far end of the read

Which head does which moves around between training runs, and some heads do none of them.
***Q15*** asks which head did what in *your* model, and how you could tell.

In [ ]:
#@title The three shapes, drawn by hand { display-mode: "form" }
# Not from a model. These are the patterns worth recognising in your own grids below.
n = 24
q = np.arange(n)
shapes = {
    "local\neach position looks at\nitself and its neighbours":
        np.exp(-((q[:, None] - q[None, :]) ** 2) / 2.0),
    "one fixed place\nevery position looks\nat the same spot":
        np.tile((q == 0).astype(float), (n, 1)),
    "the mirror\neach position looks at\nthe far end of the read":
        (q[:, None] == (n - 1 - q)[None, :]).astype(float),
}
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
for a, (title, M) in zip(axes, shapes.items()):
    a.imshow(M, cmap="magma", vmin=0, vmax=1)
    a.set_title(title, fontsize=8)
    a.set_xlabel("position looked at", fontsize=8)
    a.set_xticks([0, n - 1]); a.set_yticks([0, n - 1]); a.tick_params(labelsize=7)
axes[0].set_ylabel("position being computed", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
#@title Attention grids, one panel per head { display-mode: "form" }
# uses the `model` you trained in Section 3
while True:
    sig, lab, fwd = make_batch(1)
    if fwd[0] == 0: break

model.eval()
with torch.no_grad():
    x = model.tokenizer(sig.to(DEVICE))
    x = model.pos(x) if model.pos is not None else x
    grids = []
    for blk in model.blocks:
        _, att = blk.attn(blk.ln1(x), return_attn=True)
        grids.append(att[0].cpu().numpy())        # (n_heads, READ_LEN, READ_LEN)
        x = blk(x)

n_blocks, n_heads = len(grids), grids[0].shape[0]
fig, axes = plt.subplots(n_blocks, n_heads, squeeze=False,
                         figsize=(2.2 * n_heads, 2.3 * n_blocks))
ticks = [0, READ_LEN // 2, READ_LEN - 1]
for b in range(n_blocks):
    for h in range(n_heads):
        a = axes[b][h]
        # scale each panel to its own peak, or a diffuse head renders as solid black
        a.imshow(grids[b][h], cmap="magma", vmin=0, vmax=max(float(grids[b][h].max()), .15))
        a.set_title(f"block {b}, head {h}", fontsize=8)
        a.set_xticks(ticks); a.set_yticks(ticks); a.tick_params(labelsize=6)
        if h == 0: a.set_ylabel("position computed", fontsize=7)
        if b == n_blocks - 1: a.set_xlabel("position looked at", fontsize=7)
plt.tight_layout(); plt.show()